# Focus Guard — emotion model v2

One notebook, two machines. On the **Mac** run as-is. On the **GPU VM** set
`MIXED_PRECISION = True` and `EPOCHS = 150` in the config cell.

Order matters — each section answers a question before the next one acts on it:

1. **Baseline** — what does the current shipped model actually score?
2. **Skew** — how much does the app's CLAHE preprocessing cost a model that never trained on it?
3. **Fix** — rebuild the dataset the way the webcam sends it
4. **Train** — honest validation split, selected on the 4 states the app consumes
5. **Compare & export** — only ship it if it beats the baseline

In [ ]:
import os, sys, math, glob, json, time
import numpy as np

PROJECT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(PROJECT); sys.path.insert(0, PROJECT)

SEED, IMG, BATCH = 1337, 48, 256
EPOCHS          = 60          # VM with GPU: 150
MIXED_PRECISION = False       # VM with CUDA GPU: True
RAW_DIR, CLAHE_DIR = 'data', 'data_clahe'
BASELINE = 'src/models/focus_guard_final.h5'
BEST     = 'src/models/focus_guard_v2.h5'

CLASSES  = ['angry','disgust','fear','happy','neutral','sad','surprise']   # alphabetical = keras order
STATES   = ['FOCUS','HAPPY','STRESS','DISTRACTION']
STATE_ID = {'neutral':0,'happy':1,'angry':2,'disgust':2,'fear':2,'sad':2,'surprise':3}

print('project :', PROJECT)
print('classes :', CLASSES)

In [ ]:
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(SEED)
if MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

print('tensorflow', tf.__version__)
print('GPUs      ', tf.config.list_physical_devices('GPU') or 'none (CPU only)')
print('policy    ', keras.mixed_precision.global_policy().name)

## 1. Baseline — the number to beat

`data/test` is the **final exam**: used here to score, never to pick a checkpoint.
The 4-state score is the one that matters — `main.py` collapses 7 emotions into 4 states.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

def load_split(directory, batch=256):
    ds = keras.utils.image_dataset_from_directory(
        directory, label_mode='int', color_mode='grayscale',
        image_size=(IMG, IMG), batch_size=batch, shuffle=False)
    y = np.concatenate([b.numpy() for _, b in ds])
    return ds.map(lambda a, b: a / 255.0), y

def report(model, directory, name, full=True):
    x, y = load_split(directory)
    pred = model.predict(x, verbose=0).argmax(1)
    s_true = np.array([STATE_ID[CLASSES[i]] for i in y])
    s_pred = np.array([STATE_ID[CLASSES[i]] for i in pred])
    r = {'name': name,
         'acc7': float((pred == y).mean()),
         'acc4': float((s_pred == s_true).mean()),
         'f1_4': float(f1_score(s_true, s_pred, average='macro'))}
    print('===', name, '===')
    if full:
        print(classification_report(y, pred, target_names=CLASSES, digits=3, zero_division=0))
        print('4-state confusion', STATES)
        print(confusion_matrix(s_true, s_pred))
    print('7-class acc {acc7:.4f} | 4-state acc {acc4:.4f} | 4-state macro-F1 {f1_4:.4f}'.format(**r))
    return r

baseline_model = keras.models.load_model(BASELINE)
scores = [report(baseline_model, RAW_DIR + '/test', 'baseline / raw test')]

## 2. Rebuild the dataset the way the webcam sends it

`main.py:104` blurs + CLAHE-equalizes every face before inference. Training never did.
This cell writes a preprocessed copy; the next one measures what that mismatch was costing.
Takes about a minute for 35,887 images.

In [ ]:
import cv2

def build_clahe(src=RAW_DIR, dst=CLAHE_DIR):
    if os.path.isdir(dst):
        print('already exists, skipping:', dst); return
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))     # same params as preprocess_face()
    files = glob.glob(src + '/*/*/*.jpg')
    t0 = time.time()
    for i, path in enumerate(files):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        out = clahe.apply(cv2.GaussianBlur(img, (3, 3), 0))         # same order as preprocess_face()
        d = path.replace(src, dst, 1)
        os.makedirs(os.path.dirname(d), exist_ok=True)
        cv2.imwrite(d, out)
        if i % 8000 == 0:
            print(i, '/', len(files))
    print('wrote', len(files), 'files in', round(time.time() - t0, 1), 's')

build_clahe()

In [ ]:
# How much has the skew been costing? Same model, same faces, only preprocessing differs.
scores.append(report(baseline_model, CLAHE_DIR + '/test', 'baseline / CLAHE test (what the app sends)', full=False))
drop = scores[0]['f1_4'] - scores[1]['f1_4']
print('\n4-state macro-F1 lost to train/serve skew: {:+.4f}'.format(-drop))

## 3. Data pipeline

`ImageDataGenerator` decodes JPEGs one at a time in Python and starves the GPU. Cached
tensors are ~320 MB, so the whole dataset lives in RAM after epoch 1.

Validation is split out of `data/train` — `data/test` stays untouched until section 5.

In [ ]:
AUTO = tf.data.AUTOTUNE

train_raw, val_raw = keras.utils.image_dataset_from_directory(
    CLAHE_DIR + '/train', label_mode='categorical', color_mode='grayscale',
    image_size=(IMG, IMG), batch_size=None,
    validation_split=0.1, subset='both', seed=SEED)

aug = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.03, fill_mode='nearest'),        # ~10 degrees, not 20
    keras.layers.RandomTranslation(0.08, 0.08, fill_mode='nearest'),
    keras.layers.RandomZoom(0.1, fill_mode='nearest'),
    keras.layers.RandomContrast(0.15),
], name='aug')

def pipe(ds, training):
    ds = ds.map(lambda a, b: (a / 255.0, b), num_parallel_calls=AUTO).cache()
    if training:
        ds = ds.shuffle(10000, seed=SEED)
    ds = ds.batch(BATCH)
    if training:
        ds = ds.map(lambda a, b: (aug(a, training=True), b), num_parallel_calls=AUTO)
    return ds.prefetch(AUTO)

train_ds, val_ds = pipe(train_raw, True), pipe(val_raw, False)

counts = np.array([len(os.listdir(CLAHE_DIR + '/train/' + c)) for c in CLASSES], dtype=np.float64)
w = (counts.sum() / (len(counts) * counts)) ** 0.5                 # sqrt-balanced, not 'balanced'
class_weight = {i: float(w[i]) for i in range(len(CLASSES))}
print(dict(zip(CLASSES, counts.astype(int))))
print({CLASSES[i]: round(v, 2) for i, v in class_weight.items()})

## 4. Model and training

Two changes to the architecture: dropout before the head, and a **float32 softmax** —
under mixed precision a float16 softmax makes the loss go NaN.

In [ ]:
from tensorflow.keras.layers import (Input, Conv2D, SeparableConv2D, MaxPooling2D,
                                     BatchNormalization, Activation, GlobalAveragePooling2D,
                                     Add, Dense, Dropout)

def build_model_v2(num_classes=7, dropout=0.3):
    inp = Input(shape=(IMG, IMG, 1))
    x = Activation('relu')(BatchNormalization()(Conv2D(32, 3, strides=2, use_bias=False, padding='same')(inp)))
    x = Activation('relu')(BatchNormalization()(Conv2D(64, 3, use_bias=False, padding='same')(x)))

    for filters in (128, 256, 512):
        residual = BatchNormalization()(Conv2D(filters, 1, strides=2, padding='same', use_bias=False)(x))
        x = Activation('relu')(x)
        x = Activation('relu')(BatchNormalization()(SeparableConv2D(filters, 3, padding='same', use_bias=False)(x)))
        x = BatchNormalization()(SeparableConv2D(filters, 3, padding='same', use_bias=False)(x))
        x = MaxPooling2D(3, strides=2, padding='same')(x)
        x = Add()([x, residual])

    x = Activation('relu')(BatchNormalization()(SeparableConv2D(512, 3, padding='same', use_bias=False)(x)))
    x = Activation('relu')(BatchNormalization()(SeparableConv2D(1024, 3, padding='same', use_bias=False)(x)))
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout)(x)
    out = Dense(num_classes, activation='softmax', dtype='float32')(x)   # float32 head
    return keras.Model(inp, out)

steps = math.ceil(counts.sum() * 0.9 / BATCH)
sched = keras.optimizers.schedules.CosineDecay(
    0.0, decay_steps=EPOCHS * steps, alpha=0.02, warmup_target=1e-3, warmup_steps=5 * steps)

model = build_model_v2()
model.compile(optimizer=keras.optimizers.AdamW(sched, weight_decay=1e-4),
              loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
              metrics=['accuracy'])
print(model.count_params(), 'params |', steps, 'steps/epoch')

In [ ]:
class StateF1(keras.callbacks.Callback):
    # Selects checkpoints on the 4 states the app consumes, not on 7-class accuracy.
    def __init__(self, ds, path):
        super().__init__()
        self.ds, self.path, self.best = ds, path, -1.0
        y = np.concatenate([b.numpy().argmax(1) for _, b in ds])
        self.y = np.array([STATE_ID[CLASSES[i]] for i in y])

    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.ds, verbose=0).argmax(1)
        f1 = f1_score(self.y, [STATE_ID[CLASSES[i]] for i in p], average='macro')
        logs['val_state_f1'] = f1
        if f1 > self.best:
            self.best = f1
            self.model.save(self.path)
            print('  new best 4-state macro-F1 {:.4f} -> {}'.format(f1, self.path))

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight,
    callbacks=[
        StateF1(val_ds, BEST),                                      # MUST be before EarlyStopping
        keras.callbacks.EarlyStopping(monitor='val_state_f1', mode='max',
                                      patience=25, restore_best_weights=True, verbose=1),
        keras.callbacks.TensorBoard('runs/v2'),
    ])

## 5. Compare, then export

The v2 model is scored on the CLAHE test set because that is what the webcam feeds it.
Ship only if `f1_4` beats the baseline's CLAHE score from section 2.

In [ ]:
best = keras.models.load_model(BEST)
scores.append(report(best, CLAHE_DIR + '/test', 'v2 / CLAHE test'))

print()
for s in scores:
    print('{name:42s} acc7 {acc7:.4f}  acc4 {acc4:.4f}  F1_4 {f1_4:.4f}'.format(**s))
gain = scores[-1]['f1_4'] - scores[1]['f1_4']
print('\nv2 vs baseline on the app pipeline: {:+.4f} 4-state macro-F1'.format(gain))
json.dump(scores, open('runs/scores.json', 'w'), indent=2)

In [ ]:
# TFLite export. Run this in a FRESH kernel with MIXED_PRECISION = False,
# then copy src/models/focus_guard.tflite to the Mac.
m = keras.models.load_model(BEST)
tfl = tf.lite.TFLiteConverter.from_keras_model(m).convert()
open('src/models/focus_guard.tflite', 'wb').write(tfl)
print('wrote src/models/focus_guard.tflite', round(len(tfl) / 1e6, 2), 'MB')